# 01 | Ingestão: Landing (CSV) → Bronze (Parquet)

Lê os 3 CSVs do State of Data Brazil (2023, 2024-2025 e 2025-2026) exatamente como vieram do Kaggle e grava uma cópia fiel em **Parquet/Snappy** na camada **Bronze**, registrando as tabelas no **Glue Data Catalog** (`sod_bronze`).

Na Bronze nenhum valor é alterado: só sanitizamos o nome das colunas (o Catalog/Athena não aceitam parênteses, vírgulas e acentos) e adicionamos colunas de linhagem (`_edicao`, `_arquivo_origem`, `_data_ingestao`).

**Como rodar no AWS Academy:** AWS Glue → *Notebooks* → *Upload notebook* → IAM role **LabRole**. Ajuste a variável `BUCKET` na célula de parâmetros e execute as células em ordem.

> Este notebook tem o mesmo código do script `glue_jobs/job_01_bronze.py`, que é a versão agendável como **Glue Job** (Spark). A primeira célula (magics `%`) só tem efeito no Glue Interactive Session; localmente ela é ignorada.

### Configuração da sessão Glue (Interactive Session)

In [1]:
# Executar apenas no AWS Glue Notebook (comente esta célula ao rodar localmente)
# %idle_timeout 60
# %glue_version 5.0
# %worker_type G.1X
# %number_of_workers 2

### Parâmetros e sessão Spark

In [2]:
# Parâmetros e sessão Spark (funciona no Glue Notebook e localmente)
import os, sys
try:
    from awsglue.context import GlueContext
    from pyspark.context import SparkContext
    IS_GLUE = True
except ImportError:
    IS_GLUE = False

BUCKET = "lab-SEU-ID-AQUI"   # <-- AJUSTE: bucket criado no AWS Academy (ex.: lab-123456789012)

if IS_GLUE:
    glue_context = GlueContext(SparkContext.getOrCreate())
    spark = glue_context.spark_session
    BASE = f"s3://{BUCKET}"
else:
    from pyspark.sql import SparkSession
    ROOT = os.path.abspath("..")          # notebook roda a partir de notebooks/
    spark = (SparkSession.builder.appName("tech-challenge-fase3").master("local[*]")
             .config("spark.sql.warehouse.dir", os.path.join(ROOT, "data_lake", "_warehouse"))
             .config("spark.ui.showConsoleProgress", "false")
             .getOrCreate())
    spark.sparkContext.setLogLevel("ERROR")
    BASE = os.path.join(ROOT, "data_lake")
print("Ambiente:", "AWS Glue" if IS_GLUE else "local", "| BASE =", BASE)

26/11/02 17:06:55 WARN Utils: Your hostname, vm resolves to a loopback address: 127.0.0.1; using 192.0.2.2 instead (on interface eth0)
26/11/02 17:06:55 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/11/02 17:06:56 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Ambiente: local | BASE = /home/claude/tc3/data_lake


### Bibliotecas

In [3]:
import os
import re
import sys
import unicodedata
from datetime import datetime

from pyspark.sql import SparkSession, functions as F

# ---------------------------------------------------------------------------
# Ambiente: AWS Glue ou local
# ---------------------------------------------------------------------------

In [4]:
DB_BRONZE = "sod_bronze"

# Edição -> arquivo na landing zone
EDICOES = {
    "2023": "state_of_data_2023.csv",
    "2024_2025": "state_of_data_2024_2025.csv",
    "2025_2026": "state_of_data_2025_2026.csv",
}

In [5]:
def landing_path(edicao, arquivo):
    if IS_GLUE:
        return f"{BASE}/landing/state_of_data/edicao={edicao}/{arquivo}"
    return os.path.join(ROOT, "data", "raw", arquivo)

In [6]:
def sanitizar_nome(nome: str) -> str:
    """Converte o cabeçalho original num nome de coluna aceito por Glue/Athena.
    Ex.: "('P2_h ', 'Faixa salarial')" -> "p2_h_faixa_salarial"
         "2.h_faixa_salarial"           -> "q2_h_faixa_salarial"
    """
    s = unicodedata.normalize("NFKD", nome).encode("ascii", "ignore").decode()
    s = re.sub(r"[^0-9a-zA-Z]+", "_", s).strip("_").lower()
    if not s or s[0].isdigit():
        s = "q" + s  # Athena exige crase em coluna começando com dígito
    return s[:120]

In [7]:
def nomes_unicos(nomes):
    vistos, saida = set(), []
    for n in nomes:
        base = k = sanitizar_nome(n)
        i = 1
        while k in vistos:
            i += 1
            k = f"{base}_{i}"
        vistos.add(k)
        saida.append(k)
    return saida

In [8]:
spark.sql(f"CREATE DATABASE IF NOT EXISTS {DB_BRONZE}")
data_ingestao = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

for edicao, arquivo in EDICOES.items():
    origem = landing_path(edicao, arquivo)
    df = (spark.read
          .option("header", True)
          .option("multiLine", True)      # há respostas abertas com quebra de linha
          .option("escape", '"')
          .option("encoding", "UTF-8")
          .option("inferSchema", False)   # Bronze = tudo string, fiel à origem
          .csv(origem))

    df = df.toDF(*nomes_unicos(df.columns))
    df = (df.withColumn("_edicao", F.lit(edicao))
            .withColumn("_arquivo_origem", F.lit(arquivo))
            .withColumn("_data_ingestao", F.lit(data_ingestao)))

    tabela = f"state_of_data_{edicao}"
    destino = f"{BASE}/bronze/{tabela}" if IS_GLUE else os.path.join(BASE, "bronze", tabela)

    (df.write.mode("overwrite").format("parquet")
       .option("compression", "snappy")
       .option("path", destino)
       .saveAsTable(f"{DB_BRONZE}.{tabela}"))

    print(f"[BRONZE] {DB_BRONZE}.{tabela}: {df.count()} linhas x {len(df.columns)} colunas")

[BRONZE] sod_bronze.state_of_data_2023: 5293 linhas x 402 colunas


[BRONZE] sod_bronze.state_of_data_2024_2025: 5217 linhas x 406 colunas


[BRONZE] sod_bronze.state_of_data_2025_2026: 3495 linhas x 391 colunas


### Verificação rápida

In [9]:
for t in ["state_of_data_2023", "state_of_data_2024_2025", "state_of_data_2025_2026"]:
    df = spark.read.parquet(f"{BASE}/bronze/{t}" if IS_GLUE else os.path.join(BASE, "bronze", t))
    print(t, df.count(), "linhas |", len(df.columns), "colunas | ex.:", df.columns[2:5])

state_of_data_2023 5293 linhas | 402 colunas | ex.: ['p1_a_1_faixa_idade', 'p1_b_genero', 'p1_c_cor_raca_etnia']


state_of_data_2024_2025 5217 linhas | 406 colunas | ex.: ['q1_a_idade', 'q1_a_1_faixa_idade', 'q1_b_genero']


state_of_data_2025_2026 3495 linhas | 391 colunas | ex.: ['q1_a_idade', 'q1_a_1_faixa_idade', 'q1_b_genero']
